# Skills Matching: Гибридный подход

## Особенности ноутбука:

- Гибридный подход: автоматическое обнаружение новых технологий

## Архитектура:

- Словарь данных строится на Seed-словах (базовые технологии для классификации).

- KeyBERT извлекает термины из всех текстов.

- Автоклассификация (распределение по категориям).

Но основании этого строится динамический словарь паттернов, после чего он автоматически расширяется с новыми данными на основании инфомрации из загруженных вакансий.

In [17]:
# =============================================================================
# УСТАНОВКА ЗАВИСИМОСТЕЙ
# =============================================================================

!pip install keybert sentence-transformers python-docx pandas numpy scipy matplotlib -q

import pandas as pd
import numpy as np
from scipy import stats
import re
import os
from docx import Document
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
from keybert import KeyBERT
import warnings
warnings.filterwarnings('ignore')

print("Библиотеки загружены")

Библиотеки загружены


In [18]:
# =============================================================================
# ПОДКЛЮЧЕНИЕ GOOGLE DRIVE
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

DATA_FOLDER = "/content/drive/MyDrive/vacancy-resume-matching-dataset"
VACANCIES_PATH = os.path.join(DATA_FOLDER, "5_vacancies.csv")
ANNOTATIONS_PATH = os.path.join(DATA_FOLDER, "annotations-for-the-first-30-vacancies.txt")
CV_FOLDER = os.path.join(DATA_FOLDER, "CV")

if not os.path.exists(CV_FOLDER):
    CV_FOLDER = DATA_FOLDER

print(f"Путь к данным: {DATA_FOLDER}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Путь к данным: /content/drive/MyDrive/vacancy-resume-matching-dataset


In [19]:
# =============================================================================
# ЗАГРУЗКА ДАННЫХ
# =============================================================================

def read_docx(filepath):
    try:
        doc = Document(filepath)
        text = [p.text for p in doc.paragraphs]
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    text.append(cell.text)
        return '\n'.join(text)
    except:
        return ""

# Вакансии
vacancies_df = pd.read_csv(VACANCIES_PATH)
vacancies = {i+1: {'title': row['job_title'], 'description': row['job_description']}
             for i, row in vacancies_df.iterrows()}

# Резюме
resumes = {}
for i in range(1, 31):
    path = os.path.join(CV_FOLDER, f"{i}.docx")
    if os.path.exists(path):
        resumes[i] = read_docx(path)

# Аннотации
def parse_annotations(filepath):
    with open(filepath, 'r') as f:
        content = f.read()
    arrays = [[int(x) for x in m.split(',')]
              for m in re.findall(r'\[([\d,\s]+)\]', content)
              if len(m.split(',')) == 5]
    mid = len(arrays) // 2
    return arrays[:mid], arrays[mid:]

ann1_rankings, ann2_rankings = parse_annotations(ANNOTATIONS_PATH)

print(f"Вакансий: {len(vacancies)}, Резюме: {len(resumes)}")
print(f"Аннотации: A1={len(ann1_rankings)}, A2={len(ann2_rankings)}")

Вакансий: 5, Резюме: 30
Аннотации: A1=30, A2=30


---
## 2. Инициализация KeyBERT и базовых категорий

In [20]:
# =============================================================================
# KEYBERT
# =============================================================================

kw_model = KeyBERT(model='all-MiniLM-L6-v2')
print("KeyBERT инициализирован")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyBERT инициализирован


In [21]:
# =============================================================================
# SEED-СЛОВА ДЛЯ КЛАССИФИКАЦИИ
# =============================================================================

# Это "подсказки" для автоклассификации
CATEGORY_SEEDS = {
    'programming_languages': {
        'python', 'java', 'javascript', 'typescript', 'c++', 'c#', 'csharp',
        'ruby', 'go', 'golang', 'rust', 'kotlin', 'swift', 'php', 'perl',
        'scala', 'sql', 'bash', 'powershell', 'r', 'matlab',
    },
    'frameworks': {
        'react', 'angular', 'vue', 'django', 'flask', 'fastapi', 'spring',
        'express', 'nodejs', '.net', 'dotnet', 'rails', 'laravel', 'jquery',
        'bootstrap', 'nextjs', 'svelte', 'mvc', 'wcf', 'hibernate',
    },
    'databases': {
        'mysql', 'postgresql', 'postgres', 'mongodb', 'redis', 'oracle',
        'mssql', 'sql server', 'sqlite', 'mariadb', 'cassandra', 'dynamodb',
        'elasticsearch', 'neo4j', 'firebase',
    },
    'devops_tools': {
        'docker', 'kubernetes', 'k8s', 'aws', 'azure', 'gcp', 'jenkins',
        'git', 'github', 'gitlab', 'terraform', 'ansible', 'ci/cd', 'nginx',
    },
    'operating_systems': {
        'linux', 'unix', 'windows', 'macos', 'ubuntu', 'centos', 'redhat',
    },
    'methodologies': {
        'agile', 'scrum', 'kanban', 'devops', 'tdd', 'oop', 'solid',
    },
}

print(f" Категорий: {len(CATEGORY_SEEDS)}")
print(f" Всего seed-слов: {sum(len(s) for s in CATEGORY_SEEDS.values())}")

 Категорий: 6
 Всего seed-слов: 84


---
## 3. Автоматическое построение словаря из данных

In [22]:
# =============================================================================
# ФУНКЦИЯ ПОСТРОЕНИЯ ДИНАМИЧЕСКОГО СЛОВАРЯ
# =============================================================================

def build_vocabulary_from_data(texts, category_seeds, kw_model):
    """
    Автоматически строит словарь паттернов из текстов:
    1. KeyBERT извлекает ключевые слова
    2. Классифицируем по seed-словам
    3. Создаём regex-паттерны
    """

    # Извлекаем все ключевые слова
    all_keywords = defaultdict(list)

    print("Извлечение ключевых слов...")
    for text in texts:
        if not text:
            continue
        try:
            kws = kw_model.extract_keywords(
                text, keyphrase_ngram_range=(1, 2),
                stop_words='english', top_n=30,
                use_mmr=True, diversity=0.5
            )
            for kw, score in kws:
                all_keywords[kw.lower()].append(score)
        except:
            pass

    print(f"   Найдено уникальных терминов: {len(all_keywords)}")

    # Инициализируем словарь
    vocabulary = {cat: {} for cat in category_seeds.keys()}
    vocabulary['other_skills'] = {}

    # Функция классификации
    def classify_term(term):
        term_lower = term.lower()
        for cat, seeds in category_seeds.items():
            if term_lower in seeds:
                return cat
            for seed in seeds:
                if seed in term_lower or term_lower in seed:
                    return cat
        return None

    # Классифицируем термины
    for term, scores in all_keywords.items():
        avg_score = np.mean(scores)
        freq = len(scores)

        # Фильтруем: минимум 2 вхождения или высокий скор
        if freq < 2 and avg_score < 0.4:
            continue
        if len(term) < 2:
            continue

        category = classify_term(term)
        pattern = r'\b' + re.escape(term) + r'\b'

        if category:
            vocabulary[category][term] = {
                'pattern': pattern,
                'score': avg_score,
                'freq': freq,
                'source': 'auto'
            }
        elif avg_score > 0.35:  # Только высокоскорные в other
            vocabulary['other_skills'][term] = {
                'pattern': pattern,
                'score': avg_score,
                'freq': freq,
                'source': 'auto'
            }

    # Добавляем seed-слова, которые не нашлись в текстах
    for cat, seeds in category_seeds.items():
        for seed in seeds:
            if seed not in vocabulary[cat]:
                vocabulary[cat][seed] = {
                    'pattern': r'\b' + re.escape(seed) + r'\b',
                    'score': 0.5,
                    'freq': 0,
                    'source': 'seed'
                }

    return vocabulary

# Собираем тексты
all_texts = list(resumes.values()) + [v['description'] for v in vacancies.values()]

# Строим словарь
VOCABULARY = build_vocabulary_from_data(all_texts, CATEGORY_SEEDS, kw_model)

print("\n ДИНАМИЧЕСКИЙ СЛОВАРЬ ПОСТРОЕН:")
for cat, terms in VOCABULARY.items():
    auto = sum(1 for t in terms.values() if t['source'] == 'auto')
    seed = len(terms) - auto
    print(f"   {cat}: {len(terms)} (auto: {auto}, seed: {seed})")

Извлечение ключевых слов...
   Найдено уникальных терминов: 960

 ДИНАМИЧЕСКИЙ СЛОВАРЬ ПОСТРОЕН:
   programming_languages: 163 (auto: 143, seed: 20)
   frameworks: 20 (auto: 1, seed: 19)
   databases: 15 (auto: 0, seed: 15)
   devops_tools: 17 (auto: 3, seed: 14)
   operating_systems: 10 (auto: 3, seed: 7)
   methodologies: 7 (auto: 0, seed: 7)
   other_skills: 30 (auto: 30, seed: 0)


In [23]:
# =============================================================================
# ПРОСМОТР АВТОМАТИЧЕСКИ НАЙДЕННЫХ ТЕРМИНОВ
# =============================================================================

for cat, terms in VOCABULARY.items():
    auto_terms = [(t, info) for t, info in terms.items() if info['source'] == 'auto']
    if auto_terms:
        print(f"\n{cat}:")
        sorted_terms = sorted(auto_terms, key=lambda x: -x[1]['score'])[:5]
        for term, info in sorted_terms:
            print(f"  {term:<20} score={info['score']:.2f}, freq={info['freq']}")


programming_languages:
  golang developer     score=0.74, freq=1
  developer linux      score=0.67, freq=1
  develop frontend     score=0.61, freq=1
  android developer    score=0.61, freq=1
  frontend development score=0.59, freq=2

frameworks:
  vue                  score=0.28, freq=2

devops_tools:
  aws development      score=0.48, freq=1
  jenkins languages    score=0.46, freq=1
  bitbucket github     score=0.22, freq=2

operating_systems:
  skills linux         score=0.66, freq=1
  knowledge unix       score=0.48, freq=1
  svn unix             score=0.42, freq=1

other_skills:
  development web      score=0.53, freq=1
  backend development  score=0.52, freq=1
  development maintenance score=0.51, freq=1
  jdbc                 score=0.50, freq=3
  qualifications       score=0.50, freq=1


---
## 4. Гибридное извлечение навыков

In [24]:
# =============================================================================
# ФУНКЦИЯ ИЗВЛЕЧЕНИЯ НАВЫКОВ
# =============================================================================

def extract_skills(text, vocabulary):
    """Извлекает навыки по паттернам из словаря."""
    text_lower = text.lower()
    found = {cat: set() for cat in vocabulary.keys()}

    for cat, terms in vocabulary.items():
        for term, info in terms.items():
            if re.search(info['pattern'], text_lower, re.IGNORECASE):
                found[cat].add(term)

    return found

# Извлекаем навыки
vacancy_skills = {v_id: extract_skills(v['description'], VOCABULARY) for v_id, v in vacancies.items()}
resume_skills = {cv_id: extract_skills(text, VOCABULARY) for cv_id, text in resumes.items() if text}

print(f"Извлечены навыки: {len(vacancy_skills)} вакансий, {len(resume_skills)} резюме")

# Пример
print(f"\nПример V1 ({vacancies[1]['title']}):")
for cat, skills in vacancy_skills[1].items():
    if skills:
        print(f"   {cat}: {skills}")

Извлечены навыки: 5 вакансий, 30 резюме

Пример V1 (Software Developer - .Net):
   programming_languages: {'understanding requirements', 'javascript', 'express', 'software development', 'programming'}
   frameworks: {'angular', 'mvc', 'express', 'wcf', '.net', 'jquery'}
   databases: {'mssql'}
   methodologies: {'scrum', 'agile', 'oop', 'solid'}
   other_skills: {'applications maintain'}


---
## 5. Корреляционный анализ и веса

In [25]:
# =============================================================================
# РАСЧЕТ ПОКРЫТИЯ И ПОСТРОЕНИЕ МАТРИЦЫ
# =============================================================================

def calc_coverage(cv_skills, vac_skills):
    coverage = {}
    for cat in cv_skills.keys():
        cv = cv_skills.get(cat, set())
        vac = vac_skills.get(cat, set())
        coverage[cat] = len(cv & vac) / len(vac) if vac else 1.0
    return coverage

# Строим матрицу
feature_names = list(VOCABULARY.keys())
n_cvs = min(len(resume_skills), len(ann1_rankings))

rows = []
for cv_id in range(1, n_cvs + 1):
    if cv_id not in resume_skills:
        continue
    for vac_id in range(1, 6):
        cov = calc_coverage(resume_skills[cv_id], vacancy_skills[vac_id])
        row = {'cv_id': cv_id, 'vac_id': vac_id,
               'ann1_rank': ann1_rankings[cv_id-1][vac_id-1],
               'ann2_rank': ann2_rankings[cv_id-1][vac_id-1]}
        row.update(cov)
        rows.append(row)

df = pd.DataFrame(rows)
df['ann1_score'] = 6 - df['ann1_rank']
df['ann2_score'] = 6 - df['ann2_rank']

print(f"Матрица: {len(df)} строк, признаки: {feature_names}")

Матрица: 150 строк, признаки: ['programming_languages', 'frameworks', 'databases', 'devops_tools', 'operating_systems', 'methodologies', 'other_skills']


In [26]:
# =============================================================================
# КОРРЕЛЯЦИОННЫЙ АНАЛИЗ
# =============================================================================

correlations = {}

print(f"{'Признак':<25} {'r(A1)':>10} {'p(A1)':>10} {'r(A2)':>10} {'p(A2)':>10}")
print("-"*70)

for f in feature_names:
    r1, p1 = stats.spearmanr(df[f], df['ann1_score'])
    r2, p2 = stats.spearmanr(df[f], df['ann2_score'])
    correlations[f] = {'r1': r1, 'p1': p1, 'r2': r2, 'p2': p2}

    sig1 = '*' if p1 < 0.1 else ''
    sig2 = '*' if p2 < 0.1 else ''
    print(f"{f:<25} {r1:>+9.3f}{sig1} {p1:>9.3f} {r2:>+9.3f}{sig2} {p2:>9.3f}")

print("* p < 0.1")

Признак                        r(A1)      p(A1)      r(A2)      p(A2)
----------------------------------------------------------------------
programming_languages        +0.066     0.421    +0.088     0.285
frameworks                   -0.353*     0.000    -0.105     0.201
databases                    +0.210*     0.010    +0.100     0.225
devops_tools                 +0.252*     0.002    -0.074     0.368
operating_systems            -0.024     0.767    +0.092     0.260
methodologies                -0.094     0.250    -0.062     0.448
other_skills                 +0.277*     0.001    +0.075     0.359
* p < 0.1


In [27]:
# =============================================================================
# РАСЧЕТ ВЕСОВ ПРИЗНАКОВ
# =============================================================================

def calc_weights(correlations, target='avg'):
    weights = {}
    for f, c in correlations.items():
        if target == 'ann1':
            r, p = c['r1'], c['p1']
        elif target == 'ann2':
            r, p = c['r2'], c['p2']
        else:
            r = (c['r1'] + c['r2']) / 2
            p = min(c['p1'], c['p2'])

        weights[f] = r if (p < 0.1 and r > 0) else 0

    total = sum(weights.values())
    return {f: w/total for f, w in weights.items()} if total > 0 else {f: 1/len(weights) for f in weights}

weights_avg = calc_weights(correlations, 'avg')

for f, w in sorted(weights_avg.items(), key=lambda x: -x[1]):
    if w > 0:
        print(f"   {f:<25} {w:>8.1%} ⬅")
    else:
        print(f"   {f:<25} {w:>8.1%}")

   other_skills                 42.0% ⬅
   databases                    36.9% ⬅
   devops_tools                 21.2% ⬅
   programming_languages         0.0%
   frameworks                    0.0%
   operating_systems             0.0%
   methodologies                 0.0%


---
## 6. Matching и оценка

In [28]:
# =============================================================================
# MATCHING
# =============================================================================

def calc_score(cv_skills, vac_skills, weights):
    cov = calc_coverage(cv_skills, vac_skills)
    return sum(cov.get(f, 0) * w for f, w in weights.items())

def build_rankings(resume_skills, vacancy_skills, weights):
    rankings = {}
    for cv_id in resume_skills:
        scores = {v_id: calc_score(resume_skills[cv_id], vacancy_skills[v_id], weights)
                  for v_id in vacancy_skills}
        rankings[cv_id] = sorted(scores.keys(), key=lambda v: scores[v], reverse=True)
    return rankings

model_rankings = build_rankings(resume_skills, vacancy_skills, weights_avg)
print(f"Ранкинги построены для {len(model_rankings)} CV")

Ранкинги построены для 30 CV


In [29]:
# =============================================================================
# ОЦЕНКА РЕЗУЛЬТАТОВ ГИБРИДНОГО ПОДХОДА
# =============================================================================

def ann_to_order(ann):
    return [v for _, v in sorted(zip(ann, range(1, 6)))]

def order_to_ann(order):
    ann = [0]*5
    for i, v in enumerate(order, 1):
        ann[v-1] = i
    return ann

def evaluate(rankings, ann_rankings):
    n = min(len(rankings), len(ann_rankings))
    t1, t2, t3, taus = 0, 0, 0, []

    for i, cv_id in enumerate(sorted(rankings.keys())[:n]):
        pred = rankings[cv_id]
        true = ann_to_order(ann_rankings[i])

        if pred[0] == true[0]: t1 += 1
        if true[0] in pred[:2]: t2 += 1
        if true[0] in pred[:3]: t3 += 1

        tau, _ = stats.kendalltau(order_to_ann(pred), ann_rankings[i])
        if not np.isnan(tau): taus.append(tau)

    return {'top1': t1/n, 'top2': t2/n, 'top3': t3/n, 'tau': np.mean(taus)}

m1 = evaluate(model_rankings, ann1_rankings)
m2 = evaluate(model_rankings, ann2_rankings)

ann1_orders = {i+1: ann_to_order(r) for i, r in enumerate(ann1_rankings)}
iaa = evaluate(ann1_orders, ann2_rankings)


print(f"{'Сравнение':<35} {'Top-1':>10} {'Top-2':>10} {'Top-3':>10} {'Kendall':>10}")
print(f"{'Hybrid v3 vs Аннотатор 1':<35} {m1['top1']:>9.1%} {m1['top2']:>9.1%} {m1['top3']:>9.1%} {m1['tau']:>+9.3f}")
print(f"{'Hybrid v3 vs Аннотатор 2':<35} {m2['top1']:>9.1%} {m2['top2']:>9.1%} {m2['top3']:>9.1%} {m2['tau']:>+9.3f}")
print(f"{'IAA (A1 vs A2)':<35} {iaa['top1']:>9.1%} {iaa['top2']:>9.1%} {iaa['top3']:>9.1%} {iaa['tau']:>+9.3f}")
print(f"{'Random Baseline':<35} {'20.0%':>10} {'40.0%':>10} {'60.0%':>10} {'+0.000':>10}")

Сравнение                                Top-1      Top-2      Top-3    Kendall
Hybrid v3 vs Аннотатор 1                20.0%     50.0%     73.3%    +0.326
Hybrid v3 vs Аннотатор 2                 3.3%     20.0%     43.3%    +0.073
IAA (A1 vs A2)                          16.7%     40.0%     50.0%    +0.127
Random Baseline                          20.0%      40.0%      60.0%     +0.000


---
## 7. Демонстрация автообнаружения новых технологий

In [30]:
# =============================================================================
# ДЕМО: НОВАЯ ВАКАНСИЯ С НОВЫМИ ТЕХНОЛОГИЯМИ
# =============================================================================

new_vacancy = """
Looking for a Senior Developer with:
- Rust programming language
- WebAssembly (WASM)
- Deno runtime
- SurrealDB database
- Tauri framework
- Mojo language
"""

print("НОВАЯ ВАКАНСИЯ:")
print(new_vacancy)

# Извлекаем с текущим словарём
skills1 = extract_skills(new_vacancy, VOCABULARY)

# Добавляем новую вакансию и перестраиваем словарь
print("\n РЕЗУЛЬТАТ С ТЕКУЩИМ СЛОВАРЁМ:")
for cat, s in skills1.items():
    if s:
        print(f"   {cat}: {s}")

# KeyBERT для новых терминов
print("\n KeyBERT НАХОДИТ ДОПОЛНИТЕЛЬНО:")
new_kws = kw_model.extract_keywords(new_vacancy, top_n=10, keyphrase_ngram_range=(1,2))
for kw, score in new_kws:
    if score > 0.3:
        print(f"   {kw}: {score:.2f}")

print("\n ВЫВОД: Гибридный подход автоматически обнаруживает новые технологии!")

НОВАЯ ВАКАНСИЯ:

Looking for a Senior Developer with:
- Rust programming language
- WebAssembly (WASM)
- Deno runtime
- SurrealDB database
- Tauri framework
- Mojo language


 РЕЗУЛЬТАТ С ТЕКУЩИМ СЛОВАРЁМ:
   programming_languages: {'rust', 'developer', 'programming'}

 KeyBERT НАХОДИТ ДОПОЛНИТЕЛЬНО:
   developer rust: 0.60
   senior developer: 0.53
   rust programming: 0.46
   runtime surrealdb: 0.44
   developer: 0.43
   looking senior: 0.42
   surrealdb: 0.41
   rust: 0.38
   surrealdb database: 0.37
   framework mojo: 0.37

 ВЫВОД: Гибридный подход автоматически обнаруживает новые технологии!


---
## 8. Экспорт

In [31]:
# =============================================================================
# ЭКСПОРТ
# =============================================================================

# Словарь
vocab_rows = [{'category': cat, 'term': t, 'score': info['score'],
               'freq': info['freq'], 'source': info['source']}
              for cat, terms in VOCABULARY.items()
              for t, info in terms.items()]
pd.DataFrame(vocab_rows).to_csv('hybrid_vocabulary.csv', index=False)

# Результаты
results = [['Hybrid v3 vs A1', m1['top1'], m1['top3'], m1['tau']],
           ['Hybrid v3 vs A2', m2['top1'], m2['top3'], m2['tau']],
           ['IAA', iaa['top1'], iaa['top3'], iaa['tau']]]
pd.DataFrame(results, columns=['Model', 'Top-1', 'Top-3', 'Kendall']).to_csv('hybrid_results.csv', index=False)

print("Сохранено: hybrid_vocabulary.csv, hybrid_results.csv")

from google.colab import files
files.download('hybrid_vocabulary.csv')
files.download('hybrid_results.csv')

Сохранено: hybrid_vocabulary.csv, hybrid_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>